In [ ]:
# =========== INSTALLASI LIBRARY TAMBAHAN ===========
!pip install plotly numpy pandas matplotlib -q

# =========== IMPORT LIBRARY ===========
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
from collections import defaultdict
import itertools

In [ ]:
# =========== KONFIGURASI DATA ===========
class Container:
    def __init__(self, length=6, width=5, height=5, max_weight=50):
        self.length = length
        self.width = width
        self.height = height
        self.max_weight = max_weight
        self.volume = length * width * height
        # Batas pusat massa (dalam satuan panjang dari tengah)
        self.cog_limit_x = length * 0.1  # 10% dari panjang
        self.cog_limit_y = width * 0.1   # 10% dari lebar
        self.cog_limit_z = height * 0.15 # 15% dari tinggi (lebih toleran)

class Package:
    def __init__(self, id_, length, width, height, weight):
        self.id = id_
        self.original = (length, width, height)
        self.weight = weight
        self.volume = length * width * height
        self.orientations = self.generate_orientations()

    def generate_orientations(self):
        l, w, h = self.original
        return {
            1: (l, w, h),   # L,W,H
            2: (l, h, w),   # L,H,W
            3: (w, l, h),   # W,L,H
            4: (w, h, l),   # W,H,L
            5: (h, l, w),   # H,L,W
            6: (h, w, l)    # H,W,L
        }

# =========== FUNGSI BOTTOM-LEFT-FILL ===========
def bottom_left_fill(chromosome, container, packages_dict):
    """Menempatkan paket dengan algoritma Bottom-Left-Fill dengan pengecekan kendala"""
    positions = []
    placed = []
    space_grid = np.zeros((container.length, container.width, container.height), dtype=int)

    # Untuk tracking pusat massa
    total_mass = 0
    total_mass_x = 0
    total_mass_y = 0
    total_mass_z = 0

    for gene in chromosome:
        p_id = gene[0]
        orientation = gene[1]
        package = packages_dict[p_id]
        dims = package.orientations[orientation]
        placed_flag = False

        # Cari semua posisi yang mungkin (z terendah dulu, lalu y, lalu x)
        for z in range(container.height - dims[2] + 1):
            for y in range(container.width - dims[1] + 1):
                for x in range(container.length - dims[0] + 1):
                    # Cek apakah space kosong
                    if np.all(space_grid[x:x+dims[0], y:y+dims[1], z:z+dims[2]] == 0):

                        # =========== CEK STABILITAS (B5) ===========
                        stability_valid = True
                        if z > 0:  # Tidak di lantai dasar
                            support_area = 0
                            total_area = dims[0] * dims[1]

                            # Hitung area yang ditopang oleh kargo di bawahnya
                            for xp in range(x, x + dims[0]):
                                for yp in range(y, y + dims[1]):
                                    if space_grid[xp, yp, z-1] != 0:
                                        support_area += 1

                            # Harus minimal 50% area tertopang
                            if support_area < 0.5 * total_area:
                                stability_valid = False

                        if not stability_valid:
                            continue  # Lewati posisi ini

                        # =========== CEK KAPASITAS BEBAN (B4) ===========
                        weight_valid = True
                        if z > 0:
                            # Cari semua kargo yang menopang kargo ini
                            supporting_cargos = set()
                            for xp in range(x, x + dims[0]):
                                for yp in range(y, y + dims[1]):
                                    if space_grid[xp, yp, z-1] != 0:
                                        supporting_cargos.add(space_grid[xp, yp, z-1])

                            # Untuk setiap kargo penopang, hitung beban total di atasnya (termasuk yang akan ditempatkan)
                            for cargo_id in supporting_cargos:
                                # Dapatkan kargo penopang
                                support_pkg = packages_dict[f"P{cargo_id}"]

                                # Hitung total berat di atas cargo ini (yang sudah ditempatkan + yang akan ditempatkan)
                                weight_above = 0

                                # 1. Hitung berat dari kargo yang sudah ditempatkan di atas support_pkg
                                for pos in positions:
                                    if pos['placed'] and pos['z'] >= z:
                                        # Cek overlap vertikal dengan area support
                                        overlap_x = max(0, min(x + dims[0], pos['x'] + pos['dx']) - max(x, pos['x']))
                                        overlap_y = max(0, min(y + dims[1], pos['y'] + pos['dy']) - max(y, pos['y']))
                                        if overlap_x > 0 and overlap_y > 0:
                                            weight_above += pos['weight']

                                # 2. Tambahkan berat kargo yang akan ditempatkan
                                weight_above += package.weight

                                # Jika melebihi kapasitas + toleransi 2kg
                                if weight_above > support_pkg.weight + 2:
                                    weight_valid = False
                                    break

                        if not weight_valid:
                            continue  # Lewati posisi ini

                        # =========== TEMPATKAN PAKET ===========
                        space_grid[x:x+dims[0], y:y+dims[1], z:z+dims[2]] = int(p_id[1:])

                        # Hitung pusat massa kargo ini
                        cog_x = x + dims[0] / 2.0
                        cog_y = y + dims[1] / 2.0
                        cog_z = z + dims[2] / 2.0

                        # Update total pusat massa
                        total_mass += package.weight
                        total_mass_x += package.weight * cog_x
                        total_mass_y += package.weight * cog_y
                        total_mass_z += package.weight * cog_z

                        positions.append({
                            'id': p_id,
                            'x': x, 'y': y, 'z': z,
                            'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                            'weight': package.weight,
                            'volume': dims[0] * dims[1] * dims[2],
                            'orientation': orientation,
                            'cog_x': cog_x,
                            'cog_y': cog_y,
                            'cog_z': cog_z,
                            'placed': True
                        })
                        placed.append(p_id)
                        placed_flag = True
                        break
                if placed_flag:
                    break
            if placed_flag:
                break

        if not placed_flag:
            # Tidak bisa ditempatkan
            positions.append({
                'id': p_id,
                'x': -1, 'y': -1, 'z': -1,
                'dx': dims[0], 'dy': dims[1], 'dz': dims[2],
                'weight': package.weight,
                'volume': dims[0] * dims[1] * dims[2],
                'orientation': orientation,
                'placed': False
            })

    # Hitung pusat massa total
    if total_mass > 0:
        cog_total_x = total_mass_x / total_mass
        cog_total_y = total_mass_y / total_mass
        cog_total_z = total_mass_z / total_mass
    else:
        cog_total_x = cog_total_y = cog_total_z = 0

    # Simpan informasi pusat massa untuk digunakan di fitness
    cog_info = {
        'total_mass': total_mass,
        'cog_x': cog_total_x,
        'cog_y': cog_total_y,
        'cog_z': cog_total_z
    }

    return positions, placed, space_grid, cog_info

# =========== FUNGSI FITNESS ===========
def calculate_fitness(chromosome, container, packages_dict):
    """
    Menghitung fitness sesuai paper:
    F_i' = (E_i - (B_1 + B_2 + B_3)) • B_4 • B_5

    Dimana:
    - E = total volume muatan yang ditempatkan
    - B1, B2, B3 = penalti pusat massa (X, Y, Z)
    - B4 = 1 jika semua kendala beban terpenuhi, 0 jika tidak
    - B5 = 1 jika semua kendala stabilitas terpenuhi, 0 jika tidak
    """

    # 1. Jalankan BLF untuk mendapatkan penempatan
    positions, placed, space_grid, cog_info = bottom_left_fill(chromosome, container, packages_dict)

    # 2. Hitung total volume muatan (E)
    E = 0
    total_weight = 0
    placed_positions = [p for p in positions if p['placed']]

    for p in placed_positions:
        E += p['dx'] * p['dy'] * p['dz']
        total_weight += p['weight']

    # Jika tidak ada yang ditempatkan, fitness = 0
    if len(placed) == 0:
        return 0.0, {
            'fitness_detail': 0,
            'volume_utilization': 0,
            'penalty_cog': 0,
            'B1': 0, 'B2': 0, 'B3': 0,
            'B4': 0,
            'B5': 0,
            'total_volume': 0,
            'total_weight': 0,
            'num_placed': 0,
            'positions': positions,
            'cog_info': cog_info
        }

    # 3. Hitung penalti pusat massa (B1, B2, B3)
    # Pusat massa kontainer (tengah-tengah)
    container_center_x = container.length / 2.0
    container_center_y = container.width / 2.0
    container_center_z = container.height / 2.0

    # Pusat massa aktual
    cog_x, cog_y, cog_z = cog_info['cog_x'], cog_info['cog_y'], cog_info['cog_z']

    # Hitung deviasi pusat massa
    dev_x = abs(cog_x - container_center_x)
    dev_y = abs(cog_y - container_center_y)
    dev_z = abs(cog_z - container_center_z)

    # Penalti pusat massa (sesuai paper)
    B1 = max(0, dev_x - container.cog_limit_x)  # B1 = |cog_x - center_x| - conx
    B2 = max(0, dev_y - container.cog_limit_y)  # B2 = |cog_y - center_y| - cony
    B3 = max(0, dev_z - container.cog_limit_z)  # B3 = |cog_z - center_z| - conz

    penalty_cog = B1 + B2 + B3

    # 4. Hitung B4 (Kapasitas Beban) - sudah dicek di BLF
    # Dalam BLF, jika weight_valid = False, kargo tidak ditempatkan
    # Jadi semua kargo yang placed sudah memenuhi B4
    B4 = 1

    # Verifikasi ulang untuk memastikan
    for p in placed_positions:
        if p['z'] > 0:
            x_start, x_end = p['x'], p['x'] + p['dx']
            y_start, y_end = p['y'], p['y'] + p['dy']
            z = p['z']

            # Cari semua kargo penopang
            supporting_cargos = set()
            for x in range(x_start, x_end):
                for y in range(y_start, y_end):
                    if space_grid[x, y, z-1] != 0:
                        supporting_cargos.add(space_grid[x, y, z-1])

            # Untuk setiap kargo penopang, hitung beban di atasnya
            for cargo_id in supporting_cargos:
                support_pkg = packages_dict[f"P{cargo_id}"]
                weight_above = 0

                # Hitung total berat di atas kargo penopang
                for p2 in placed_positions:
                    if p2['z'] >= z:
                        # Cek overlap
                        overlap_x = max(0, min(p['x'] + p['dx'], p2['x'] + p2['dx']) - max(p['x'], p2['x']))
                        overlap_y = max(0, min(p['y'] + p['dy'], p2['y'] + p2['dy']) - max(p['y'], p2['y']))
                        if overlap_x > 0 and overlap_y > 0:
                            weight_above += p2['weight']

                if weight_above > support_pkg.weight + 2:
                    B4 = 0
                    break
            if B4 == 0:
                break

    # 5. Hitung B5 (Stabilitas) - sudah dicek di BLF
    # Dalam BLF, jika stability_valid = False, kargo tidak ditempatkan
    B5 = 1

    # Verifikasi ulang untuk memastikan
    for p in placed_positions:
        if p['z'] > 0:
            support_area = 0
            total_area = p['dx'] * p['dy']

            x_start, x_end = p['x'], p['x'] + p['dx']
            y_start, y_end = p['y'], p['y'] + p['dy']
            z = p['z']

            # Hitung area yang ditopang
            for x in range(x_start, x_end):
                for y in range(y_start, y_end):
                    if space_grid[x, y, z-1] != 0:
                        support_area += 1

            # Jika kurang dari 50%, B5 = 0
            if support_area < 0.5 * total_area:
                B5 = 0
                break

    # 6. Hitung fitness akhir sesuai rumus paper
    # F_i' = (E_i - (B_1 + B_2 + B_3)) • B_4 • B_5
    fitness_raw = E - penalty_cog
    fitness_final = fitness_raw * B4 * B5

    # Normalisasi ke persentase untuk kemudahan interpretasi
    volume_utilization = (E / container.volume) * 100 if container.volume > 0 else 0

    # Return fitness utama dan metadata
    return fitness_final, {
        'fitness_detail': fitness_final,
        'volume_utilization': volume_utilization,
        'penalty_cog': penalty_cog,
        'B1': B1,
        'B2': B2,
        'B3': B3,
        'B4': B4,
        'B5': B5,
        'total_volume': E,
        'total_weight': total_weight,
        'num_placed': len(placed),
        'positions': positions,
        'cog_info': cog_info,
        'cog_actual': (cog_x, cog_y, cog_z),
        'cog_deviation': (dev_x, dev_y, dev_z)
    }

# =========== FUNGSI GENETIC ALGORITHM ===========
def create_chromosome(packages_list):
    """Membuat kromosom acak"""
    ids = [p.id for p in packages_list]
    random.shuffle(ids)
    chromosome = []
    for p_id in ids:
        orientation = random.randint(1, 6)
        chromosome.append((p_id, orientation))
    return chromosome

def selection(population, fitness_scores, num_parents=2):
    """Seleksi roulette wheel"""
    # Konversi fitness negatif ke 0
    adjusted_fitness = [max(0.001, f) for f in fitness_scores]
    total_fitness = sum(adjusted_fitness)

    if total_fitness == 0:
        return random.sample(list(enumerate(population)), num_parents)

    probabilities = [f/total_fitness for f in adjusted_fitness]
    selected_indices = np.random.choice(len(population), size=num_parents,
                                        p=probabilities, replace=False)
    return [(idx, population[idx]) for idx in selected_indices]

# =========== FUNGSI PMX CROSSOVER DENGAN DETAIL ===========
def pmx_crossover(parent1, parent2):
    """Partially Matched Crossover dengan detail perhitungan"""
    size = len(parent1)
    p1 = parent1.copy()
    p2 = parent2.copy()

    print(f"\n=== DETAIL CROSSOVER ===")
    print(f"Parent 1: {parent1}")
    print(f"Parent 2: {parent2}")

    # Pilih dua titik potong acak
    cut1 = random.randint(0, size-2)
    cut2 = random.randint(cut1+1, size-1)
    print(f"\nTitik potong: cut1={cut1}, cut2={cut2}")
    print(f"Bagian tengah yang akan disalin:")
    print(f"  Parent1[{cut1}:{cut2}]: {parent1[cut1:cut2]}")
    print(f"  Parent2[{cut1}:{cut2}]: {parent2[cut1:cut2]}")

    child1 = [None] * size
    child2 = [None] * size

    # Salin bagian tengah
    child1[cut1:cut2] = p1[cut1:cut2]
    child2[cut1:cut2] = p2[cut1:cut2]
    print(f"\nInisialisasi child setelah copy bagian tengah:")
    print(f"  Child1: {child1}")
    print(f"  Child2: {child2}")

    # Mapping
    mapping1 = {}
    mapping2 = {}
    for i in range(cut1, cut2):
        mapping1[p1[i][0]] = p2[i][0]
        mapping2[p2[i][0]] = p1[i][0]

    print(f"\nMapping:")
    print(f"  Parent1 → Parent2: {mapping1}")
    print(f"  Parent2 → Parent1: {mapping2}")

    # Isi sisa posisi untuk child1
    print(f"\nMengisi posisi lainnya untuk Child1:")
    for i in range(size):
        if i < cut1 or i >= cut2:
            gene = p2[i]
            print(f"\n  Posisi {i}: Gene awal dari Parent2 = {gene}")

            step = 1
            while gene[0] in [g[0] for g in child1 if g is not None]:
                print(f"    Langkah {step}: Gene {gene[0]} sudah ada di child1")

                if gene[0] in mapping1:
                    gene_id = mapping1[gene[0]]
                    print(f"      Mapping ditemukan: {gene[0]} → {gene_id}")

                    # Cari gene dengan id ini di parent2
                    for g in p2:
                        if g[0] == gene_id:
                            old_gene = gene
                            gene = g
                            print(f"      Gene baru dari Parent2: {old_gene} → {gene}")
                            break
                else:
                    print(f"      Tidak ada mapping untuk {gene[0]}")
                    print(f"      Mencari gene dari Parent1 yang belum ada di child1")

                    for g in p1:
                        if g[0] not in [cg[0] for cg in child1 if cg is not None]:
                            old_gene = gene
                            gene = g
                            print(f"      Gene baru dari Parent1: {old_gene} → {gene}")
                            break
                step += 1

            child1[i] = gene
            print(f"    Final: child1[{i}] = {gene}")

    # Isi sisa posisi untuk child2
    print(f"\nMengisi posisi lainnya untuk Child2:")
    for i in range(size):
        if i < cut1 or i >= cut2:
            gene = p1[i]
            print(f"\n  Posisi {i}: Gene awal dari Parent1 = {gene}")

            step = 1
            while gene[0] in [g[0] for g in child2 if g is not None]:
                print(f"    Langkah {step}: Gene {gene[0]} sudah ada di child2")

                if gene[0] in mapping2:
                    gene_id = mapping2[gene[0]]
                    print(f"      Mapping ditemukan: {gene[0]} → {gene_id}")

                    # Cari gene dengan id ini di parent1
                    for g in p1:
                        if g[0] == gene_id:
                            old_gene = gene
                            gene = g
                            print(f"      Gene baru dari Parent1: {old_gene} → {gene}")
                            break
                else:
                    print(f"      Tidak ada mapping untuk {gene[0]}")
                    print(f"      Mencari gene dari Parent2 yang belum ada di child2")

                    for g in p2:
                        if g[0] not in [cg[0] for cg in child2 if cg is not None]:
                            old_gene = gene
                            gene = g
                            print(f"      Gene baru dari Parent2: {old_gene} → {gene}")
                            break
                step += 1

            child2[i] = gene
            print(f"    Final: child2[{i}] = {gene}")

    print(f"\nHasil akhir crossover:")
    print(f"  Child1: {child1}")
    print(f"  Child2: {child2}")

    return child1, child2

# =========== FUNGSI MUTASI DENGAN DETAIL ===========
def mutate(chromosome, mutation_rate=0.2):
    """Mutasi kromosom dengan detail perhitungan"""
    mutated = chromosome.copy()

    print(f"\n=== DETAIL MUTASI ===")
    print(f"Kromosom awal: {chromosome}")
    print(f"Mutation rate: {mutation_rate}")

    mutations_applied = []

    # Cek apakah swap akan terjadi
    swap_random = random.random()
    print(f"\n1. Cek swap mutation: random={swap_random:.3f}, threshold={mutation_rate}")

    if swap_random < mutation_rate:
        # Swap dua posisi acak
        i, j = random.sample(range(len(mutated)), 2)
        mutated[i], mutated[j] = mutated[j], mutated[i]
        mutations_applied.append(f"Swap posisi {i} dan {j}: {mutated[i]} ↔ {mutated[j]}")
        print(f"   SWAP DITERAPKAN: posisi {i} dan {j} ditukar")
        print(f"   Setelah swap: {mutated}")
    else:
        print(f"   Swap TIDAK diterapkan: {swap_random:.3f} >= {mutation_rate}")

    # Cek apakah orientasi akan diubah
    orient_random = random.random()
    print(f"\n2. Cek orientation mutation: random={orient_random:.3f}, threshold={mutation_rate}")

    if orient_random < mutation_rate:
        # Ubah orientasi acak
        idx = random.randint(0, len(mutated)-1)
        old_orient = mutated[idx][1]
        new_orient = random.randint(1, 6)
        mutated[idx] = (mutated[idx][0], new_orient)
        mutations_applied.append(f"Ubah orientasi posisi {idx}: {old_orient} → {new_orient}")
        print(f"   ORIENTATION MUTATION DITERAPKAN: posisi {idx}, orientasi {old_orient} → {new_orient}")
        print(f"   Setelah ubah orientasi: {mutated}")
    else:
        print(f"   Orientation mutation TIDAK diterapkan: {orient_random:.3f} >= {mutation_rate}")

    if mutations_applied:
        print(f"\nTotal mutasi yang diterapkan: {len(mutations_applied)}")
        for i, mutation in enumerate(mutations_applied, 1):
            print(f"  {i}. {mutation}")
    else:
        print(f"\nTidak ada mutasi yang diterapkan")

    print(f"\nKromosom akhir setelah mutasi: {mutated}")

    return mutated

# =========== VISUALISASI 3D ===========
def visualize_packing(positions, container_dims, title="3D Bin Packing"):
    """Visualisasi terbaik dengan kubus transparan dan garis tepi"""
    fig = go.Figure()

    # Kontainer sebagai wireframe transparan
    container_lines = [
        # Bottom square
        [0, 1], [1, 2], [2, 3], [3, 0],
        # Top square
        [4, 5], [5, 6], [6, 7], [7, 4],
        # Vertical edges
        [0, 4], [1, 5], [2, 6], [3, 7]
    ]

    container_vertices = [
        [0, 0, 0],
        [container_dims[0], 0, 0],
        [container_dims[0], container_dims[1], 0],
        [0, container_dims[1], 0],
        [0, 0, container_dims[2]],
        [container_dims[0], 0, container_dims[2]],
        [container_dims[0], container_dims[1], container_dims[2]],
        [0, container_dims[1], container_dims[2]]
    ]

    for line in container_lines:
        fig.add_trace(go.Scatter3d(
            x=[container_vertices[line[0]][0], container_vertices[line[1]][0]],
            y=[container_vertices[line[0]][1], container_vertices[line[1]][1]],
            z=[container_vertices[line[0]][2], container_vertices[line[1]][2]],
            mode='lines',
            line=dict(color='gray', width=2, dash='dash'),
            name='Container',
            showlegend=True if line == [0, 1] else False
        ))

    # Warna untuk paket
    colors = ['rgba(255, 0, 0, 0.7)',   # merah transparan
              'rgba(0, 255, 0, 0.7)',   # hijau transparan
              'rgba(0, 0, 255, 0.7)',   # biru transparan
              'rgba(255, 165, 0, 0.7)', # oranye transparan
              'rgba(128, 0, 128, 0.7)', # ungu transparan
              'rgba(255, 255, 0, 0.7)'] # kuning transparan

    for i, pos in enumerate(positions):
        if pos['x'] != -1:
            x, y, z = pos['x'], pos['y'], pos['z']
            dx, dy, dz = pos['dx'], pos['dy'], pos['dz']

            # Warna untuk paket ini
            color_idx = int(pos['id'][1:]) - 1 if pos['id'][1:].isdigit() else i
            color = colors[color_idx % len(colors)]

            # Tambahkan kubus solid (transparan)
            fig.add_trace(go.Mesh3d(
                x=[x, x+dx, x+dx, x, x, x+dx, x+dx, x],
                y=[y, y, y+dy, y+dy, y, y, y+dy, y+dy],
                z=[z, z, z, z, z+dz, z+dz, z+dz, z+dz],
                i=[0, 0, 0, 1, 1, 2],
                j=[1, 2, 4, 3, 5, 6],
                k=[2, 3, 5, 7, 6, 7],
                color=color,
                opacity=0.5,
                flatshading=True,
                name=f"{pos['id']} ({dx}×{dy}×{dz})"
            ))

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor='center'),
        scene=dict(
            xaxis_title='Panjang (X)',
            yaxis_title='Lebar (Y)',
            zaxis_title='Tinggi (Z)',
            aspectmode='data',
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        showlegend=True,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )

    return fig

# =========== ALGORITMA GENETIK UTAMA ===========
def genetic_algorithm(population, container, packages_dict, generations=3):
    """Algoritma genetik utama"""
    all_results = []
    current_pop = population.copy()

    for gen in range(generations):
        print(f"\n{'='*60}")
        print(f"GENERASI {gen}")
        print(f"{'='*60}")

        # Evaluasi fitness
        fitness_scores = []
        chrom_results = []

        for i, chrom in enumerate(current_pop):
            fitness, metadata = calculate_fitness(chrom, container, packages_dict)
            fitness_scores.append(fitness)
            chrom_results.append({
                'Generation': gen,
                'Chromosome': f"K{i+1}",
                'Sequence': chrom.copy(),
                'Fitness': fitness,
                'VolumeUtil': metadata['volume_utilization'],
                'PenaltyCog': metadata['penalty_cog'],
                'B4': metadata['B4'],
                'B5': metadata['B5'],
                'Placed': metadata['num_placed'],
                'Positions': metadata['positions'],
                'Metadata': metadata
            })
            print(f"K{i+1}: Fitness={fitness:.2f}, VU={metadata['volume_utilization']:.2f}%, Terpasang={metadata['num_placed']}/{len(chrom)}")
            print(f"  Pusat Massa: {metadata['cog_actual']}, Deviasi: {metadata['cog_deviation']}")
            print(f"  B4={metadata['B4']}, B5={metadata['B5']}, Penalti COG={metadata['penalty_cog']:.2f}")

            # Tampilkan detail perhitungan
            print(f"\n1. PENEMPATAN PAKET:")
            placed_count = 0
            for pos in metadata['positions']:
                if pos['placed']:
                    placed_count += 1
                    print(f"   {pos['id']}: Posisi({pos['x']},{pos['y']},{pos['z']}) "
                          f"Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']} "
                          f"Berat:{pos['weight']}kg Volume:{pos['dx']*pos['dy']*pos['dz']}cm³")
                else:
                    print(f"   {pos['id']}: TIDAK TERPASANG Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']}")

            print(f"\n2. PERHITUNGAN VOLUME (E):")
            print(f"   Total Volume Terpasang: {metadata['total_volume']} cm³")
            print(f"   Volume Kontainer: {container.volume} cm³")
            print(f"   Volume Utilization: {metadata['volume_utilization']:.2f}%")

            print(f"\n3. PERHITUNGAN PUSAT MASSA:")
            print(f"   Total Massa: {metadata['cog_info']['total_mass']:.1f} kg")
            print(f"   Pusat Massa Aktual: ({metadata['cog_actual'][0]:.2f}, {metadata['cog_actual'][1]:.2f}, {metadata['cog_actual'][2]:.2f})")
            print(f"   Pusat Kontainer (ideal): ({container.length/2:.1f}, {container.width/2:.1f}, {container.height/2:.1f})")
            print(f"   Deviasi: ({metadata['cog_deviation'][0]:.2f}, {metadata['cog_deviation'][1]:.2f}, {metadata['cog_deviation'][2]:.2f})")
            print(f"   Batas Maks Deviasi: ({container.cog_limit_x:.1f}, {container.cog_limit_y:.1f}, {container.cog_limit_z:.1f})")

            print(f"\n4. PENALTI PUSAT MASSA (B1, B2, B3):")
            print(f"   B1 (X): max(0, {metadata['cog_deviation'][0]:.2f} - {container.cog_limit_x:.1f}) = {metadata['B1']:.2f}")
            print(f"   B2 (Y): max(0, {metadata['cog_deviation'][1]:.2f} - {container.cog_limit_y:.1f}) = {metadata['B2']:.2f}")
            print(f"   B3 (Z): max(0, {metadata['cog_deviation'][2]:.2f} - {container.cog_limit_z:.1f}) = {metadata['B3']:.2f}")
            print(f"   Total Penalti COG (B1+B2+B3): {metadata['penalty_cog']:.2f}")

            print(f"\n5. KENDALA KAPASITAS BEBAN (B4):")
            if metadata['B4'] == 1:
                print(f"   ✓ SEMUA kargo memenuhi kapasitas beban")
                print(f"   B4 = 1")
            else:
                print(f"   ✗ ADA kargo yang kelebihan beban")
                print(f"   B4 = 0")

            print(f"\n6. KENDALA STABILITAS (B5):")
            if metadata['B5'] == 1:
                print(f"   ✓ SEMUA kargo stabil (≥50% area tertopang)")
                print(f"   B5 = 1")
            else:
                print(f"   ✗ ADA kargo tidak stabil (<50% area tertopang)")
                print(f"   B5 = 0")

            print(f"\n7. PERHITUNGAN FITNESS AKHIR:")
            print(f"   E (Volume) = {metadata['total_volume']}")
            print(f"   Total Penalti COG = {metadata['penalty_cog']:.2f}")
            print(f"   Fitness Raw = E - (B1+B2+B3) = {metadata['total_volume']} - {metadata['penalty_cog']:.2f} = {metadata['total_volume'] - metadata['penalty_cog']:.2f}")
            print(f"   Fitness Final = [E - (B1+B2+B3)] × B4 × B5")
            print(f"                  = {metadata['total_volume'] - metadata['penalty_cog']:.2f} × {metadata['B4']} × {metadata['B5']}")
            print(f"                  = {fitness:.2f}")

            print(f"\n{'─'*60}")
            print(f"KESIMPULAN K{i+1}: Fitness = {fitness:.2f}, VU = {metadata['volume_utilization']:.2f}%, Terpasang = {placed_count}/{len(chrom)}")
            print(f"{'─'*60}")

        all_results.extend(chrom_results)

        if gen == generations - 1:
            break  # Stop di generasi terakhir

        # Seleksi
        parents = selection(current_pop, fitness_scores, num_parents=2)
        parent1_idx, parent1 = parents[0]
        parent2_idx, parent2 = parents[1]

        # Crossover
        child1, child2 = pmx_crossover(parent1, parent2)

        # Mutasi
        mutate_idx = random.randint(0, len(current_pop)-1)
        mutated_child = mutate(current_pop[mutate_idx])

        # Membentuk populasi baru (elitism + anak + mutasi)
        new_population = []

        # Elitism: ambil 2 terbaik
        elite_indices = np.argsort(fitness_scores)[-2:]
        for idx in elite_indices:
            new_population.append(current_pop[idx])

        # Tambahkan anak hasil crossover
        new_population.append(child1)
        new_population.append(child2)

        # Tambahkan hasil mutasi
        new_population.append(mutated_child)

        # Jika kurang, tambahkan kromosom acak
        while len(new_population) < len(current_pop):
            new_chrom = create_chromosome(list(packages_dict.values()))
            new_population.append(new_chrom)

        current_pop = new_population[:len(population)]

    return all_results, current_pop

# =========== FUNGSI UNTUK ANALISIS DETAIL ===========
def analyze_chromosome(chromosome, container, packages_dict):
    """Analisis detail kromosom"""
    fitness, metadata = calculate_fitness(chromosome, container, packages_dict)

    print(f"\n{'='*60}")
    print(f"ANALISIS DETAIL KROMOSOM")
    print(f"{'='*60}")
    print(f"Urutan: {chromosome}")

    print(f"\nPosisi akhir paket:")
    for pos in metadata['positions']:
        if pos['placed']:
            print(f"{pos['id']}: Posisi({pos['x']},{pos['y']},{pos['z']}) Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']} Berat:{pos['weight']}kg")
        else:
            print(f"{pos['id']}: TIDAK TERPASANG Dimensi{pos['dx']}x{pos['dy']}x{pos['dz']}")

    print(f"\nPerhitungan Fitness (sesuai paper):")
    print(f"Total Volume Paket Terpasang (E): {metadata['total_volume']} cm³")
    print(f"Volume Kontainer: {container.volume} cm³")
    print(f"Volume Utilization: {metadata['volume_utilization']:.2f}%")
    print(f"Pusat Massa Aktual: {metadata['cog_actual']}")
    print(f"Deviasi Pusat Massa: {metadata['cog_deviation']}")
    print(f"Penalti Pusat Massa (B1+B2+B3): {metadata['penalty_cog']:.2f}")
    print(f"  B1 (X): {metadata['B1']:.2f}")
    print(f"  B2 (Y): {metadata['B2']:.2f}")
    print(f"  B3 (Z): {metadata['B3']:.2f}")
    print(f"B4 (Kapasitas Beban): {'LULUS' if metadata['B4'] == 1 else 'GAGAL'}")
    print(f"B5 (Stabilitas): {'LULUS' if metadata['B5'] == 1 else 'GAGAL'}")
    print(f"Fitness = (E - (B1+B2+B3)) × B4 × B5")
    print(f"        = ({metadata['total_volume']} - {metadata['penalty_cog']:.2f}) × {metadata['B4']} × {metadata['B5']}")
    print(f"        = {fitness:.2f}")

    return metadata['positions']

# =========== FUNGSI UTAMA ===========
def main():
    # =========== INISIALISASI DATA DEFAULT ===========
    print("=== INISIALISASI DATA DEFAULT ===\n")

    container = Container()
    packages = [
        Package("P1", 4, 3, 2, 8),
        Package("P2", 5, 2, 2, 12),
        Package("P3", 3, 3, 3, 10),
        Package("P4", 2, 4, 2, 5),
        Package("P5", 4, 2, 1, 3),
        Package("P6", 2, 2, 4, 7)
    ]

    print("=== KONTAINER ===")
    print(f"Dimensi: {container.length}x{container.width}x{container.height} cm")
    print(f"Volume: {container.volume} cm³")
    print(f"Berat maks: {container.max_weight} kg")
    print(f"Batas pusat massa: X={container.cog_limit_x:.1f}, Y={container.cog_limit_y:.1f}, Z={container.cog_limit_z:.1f} cm\n")

    print("=== PAKET ===")
    for p in packages:
        print(f"{p.id}: {p.original} cm, {p.weight} kg, Volume: {p.volume} cm³")

    # Buat packages dictionary untuk akses cepat
    packages_dict = {p.id: p for p in packages}

    # Inisialisasi populasi awal
    print("\n=== INISIALISASI POPULASI AWAL ===")
    population = [
        [("P1", 1), ("P2", 2), ("P6", 3), ("P4", 4), ("P5", 5), ("P3", 3)],
        [("P6", 1), ("P5", 1), ("P4", 1), ("P3", 1), ("P2", 1), ("P1", 1)],
        [("P3", 2), ("P1", 3), ("P6", 4), ("P2", 5), ("P5", 6), ("P4", 1)],
        [("P4", 6), ("P2", 4), ("P6", 3), ("P1", 2), ("P3", 5), ("P5", 1)]
    ]

    for i, chrom in enumerate(population, 1):
        print(f"K{i}: {chrom}")

    # Jalankan algoritma genetik
    print("\n" + "="*60)
    print("MEMULAI ALGORITMA GENETIK")
    print("="*60)

    random.seed(42)  # Untuk reproducibility

    all_results, final_population = genetic_algorithm(
        population, container, packages_dict, generations=3
    )

    # Tampilkan hasil akhir
    print("\n" + "="*60)
    print("HASIL AKHIR")
    print("="*60)

    # Cari solusi terbaik
    best_result = max(all_results, key=lambda x: x['Fitness'])

    print(f"\nSOLUSI TERBAIK:")
    print(f"Generasi: {best_result['Generation']}")
    print(f"Kromosom: {best_result['Chromosome']}")
    print(f"Urutan: {best_result['Sequence']}")
    print(f"Fitness: {best_result['Fitness']:.2f}")
    print(f"Volume Utilization: {best_result['VolumeUtil']:.2f}%")
    print(f"Penalti Pusat Massa: {best_result['PenaltyCog']:.2f}")
    print(f"B4 (Kapasitas Beban): {'LULUS' if best_result['B4'] == 1 else 'GAGAL'}")
    print(f"B5 (Stabilitas): {'LULUS' if best_result['B5'] == 1 else 'GAGAL'}")
    print(f"Paket Terpasang: {best_result['Placed']}/{len(packages)}")

    # Analisis detail kromosom terbaik
    print("\n" + "="*60)
    print("ANALISIS DETAIL KROMOSOM TERBAIK")
    print("="*60)
    best_positions = analyze_chromosome(best_result['Sequence'], container, packages_dict)

    # Tampilkan tabel hasil
    print(f"\n{'='*60}")
    print("TABEL HASIL LENGKAP")
    print(f"{'='*60}")

    # Buat DataFrame dari hasil
    df_results = pd.DataFrame([{
        'Generasi': r['Generation'],
        'Kromosom': r['Chromosome'],
        'Fitness': f"{r['Fitness']:.2f}",
        'VU%': f"{r['VolumeUtil']:.2f}",
        'PenCOG': f"{r['PenaltyCog']:.2f}",
        'B4': r['B4'],
        'B5': r['B5'],
        'Terpasang': f"{r['Placed']}/{len(packages)}",
        'Urutan': str(r['Sequence'])
    } for r in all_results])

    print(df_results.to_string(index=False))

    # Visualisasi kromosom terbaik
    print(f"\n{'='*60}")
    print("VISUALISASI SOLUSI TERBAIK")
    print(f"{'='*60}")

    fig = visualize_packing(best_positions,
                          (container.length, container.width, container.height),
                          f"Solusi Terbaik - Fitness: {best_result['Fitness']:.2f}")
    fig.show()

    # Export hasil ke CSV
    df_export = pd.DataFrame([{
        'Generation': r['Generation'],
        'Chromosome': r['Chromosome'],
        'Fitness': r['Fitness'],
        'VolumeUtilization': r['VolumeUtil'],
        'PenaltyCog': r['PenaltyCog'],
        'B4': r['B4'],
        'B5': r['B5'],
        'PlacedPackages': r['Placed'],
        'Sequence': str(r['Sequence'])
    } for r in all_results])

    df_export.to_csv('bin_packing_results_new.csv', index=False)
    print("\nHasil telah diexport ke 'bin_packing_results_new.csv'")

# Jalankan program
if __name__ == "__main__":
    main()